# Azure + LangChain — End-to-End RAG

Below is a **simple but production-oriented RAG skeleton** covering:

- Azure Blob Storage — document storage
- Azure Document Intelligence — document extraction
- Chunking
- Azure OpenAI embeddings
- Azure AI Search
- Hybrid search
- Semantic reranking
- Top-K
- Azure OpenAI generation
- Azure AI Content Safety
- RAGAS
- Precision / Recall
- DeepEval
- LangChain orchestration

Azure AI Search supports hybrid keyword + vector search and can apply semantic ranking on the merged results. Microsoft recommends hybrid retrieval when you want strong recall across lexical and semantic matches. 

---

# 1. Overall Architecture

```text
                         ┌──────────────┐
                         │     User     │
                         └──────┬───────┘
                                │
                                ▼
                       ┌─────────────────┐
                       │ Content Safety  │
                       │  Input Check    │
                       └────────┬────────┘
                                │
                                ▼
                         ┌─────────────┐
                         │  LangChain  │
                         └──────┬──────┘
                                │
                                ▼
                      ┌──────────────────┐
                      │ Azure AI Search  │
                      │                 │
                      │ Keyword Search  │
                      │ Vector Search   │
                      │ Hybrid Search   │
                      │ Semantic Rank   │
                      └────────┬─────────┘
                               │
                            Top-K
                               │
                               ▼
                     ┌──────────────────┐
                     │ Context Builder  │
                     └────────┬─────────┘
                              │
                              ▼
                       ┌──────────────┐
                       │ Azure OpenAI │
                       └──────┬───────┘
                              │
                              ▼
                     ┌─────────────────┐
                     │ Content Safety  │
                     │ Output Check    │
                     └────────┬────────┘
                              │
                              ▼
                           Answer
```

### Document ingestion

```text
PDF / DOCX / Images
        │
        ▼
 Azure Blob Storage
        │
        ▼
Document Intelligence
        │
        ▼
 Text + Tables + Metadata
        │
        ▼
     Chunking
        │
        ▼
 Azure OpenAI Embeddings
        │
        ▼
 Azure AI Search Index
```

---

# 2. Install Packages

```bash
pip install -U \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-text-splitters \
    azure-search-documents \
    azure-ai-contentsafety \
    azure-core \
    azure-identity \
    ragas \
    deepeval
```

For production, pin compatible package versions rather than leaving everything unpinned.

---

# 3. Environment Variables

```env
AZURE_OPENAI_ENDPOINT=https://xxx.openai.azure.com/
AZURE_OPENAI_API_KEY=xxx
AZURE_OPENAI_CHAT_DEPLOYMENT=gpt-4o
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-small

AZURE_SEARCH_ENDPOINT=https://xxx.search.windows.net
AZURE_SEARCH_KEY=xxx
AZURE_SEARCH_INDEX=documents

CONTENT_SAFETY_ENDPOINT=https://xxx.cognitiveservices.azure.com/
CONTENT_SAFETY_KEY=xxx
```

For production, prefer **Microsoft Entra ID / Managed Identity** instead of storing service keys where possible.

---

# 4. Azure OpenAI — LLM

```python
import os

from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"],
    api_version="2024-10-21",
    temperature=0
)
```

Architecture:

```text
LangChain
   ↓
AzureChatOpenAI
   ↓
Azure OpenAI
   ↓
GPT deployment
```

---

# 5. Azure OpenAI — Embeddings

```python
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
    api_version="2024-10-21"
)
```

Conceptually:

```text
Document Chunk
     ↓
Embedding Model
     ↓
Vector
     ↓
Azure AI Search
```

---

# 6. Document Loading

For a simple demo:

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("hr_policy.pdf")

documents = loader.load()

print(len(documents))
```

For enterprise PDFs containing complex layouts, scanned pages and tables, use **Azure Document Intelligence** during ingestion rather than relying solely on a basic PDF text loader.

---

# 7. Chunking

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))
```

Example:

```text
200-page PDF
     ↓
Document extraction
     ↓
Recursive / structure-aware chunking
     ↓
1000 chunks
```

### Why overlap?

If a sentence crosses the boundary:

```text
Chunk 1:
"...employees can carry forward"

Chunk 2:
"up to 5 vacation days..."
```

overlap helps preserve context.

For production, chunk size should be determined through retrieval evaluation rather than blindly choosing 500/800/1000 characters.

---

# 8. Add Metadata

```python
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["source"] = "hr_policy.pdf"
```

Better:

```python
chunk.metadata.update({
    "document_type": "HR_POLICY",
    "department": "HR",
    "country": "India"
})
```

Metadata is useful for:

- Filtering
- Security
- Citations
- Debugging
- Evaluation

---

# 9. Azure AI Search

Azure AI Search needs an index containing fields such as:

```text
id
content
contentVector
source
page
document_type
department
country
```

Conceptually:

```text
Azure AI Search Index
│
├── content
├── contentVector
├── source
├── page
├── department
└── country
```

Azure's Python SDK supports vector and hybrid search through `azure-search-documents`. 

---

# 10. Simple Azure AI Search Index

For demonstration, you can create an index like this:

```python
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile
)

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
key = os.environ["AZURE_SEARCH_KEY"]
index_name = os.environ["AZURE_SEARCH_INDEX"]

index_client = SearchIndexClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True
    ),

    SearchableField(
        name="content",
        type=SearchFieldDataType.String
    ),

    SearchableField(
        name="source",
        type=SearchFieldDataType.String,
        filterable=True
    ),

    SimpleField(
        name="page",
        type=SearchFieldDataType.Int32,
        filterable=True
    ),

    SearchableField(
        name="department",
        type=SearchFieldDataType.String,
        filterable=True
    ),

    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(
            SearchFieldDataType.Single
        ),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="my-vector-profile"
    )
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="my-hnsw"
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="my-vector-profile",
            algorithm_configuration_name="my-hnsw"
        )
    ]
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

index_client.create_or_update_index(index)
```

**Important:** `1536` must match the dimensionality of the embedding model you actually deploy.

---

# 11. Upload Chunks

```python
from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(key)
)

records = []

for i, chunk in enumerate(chunks):

    vector = embeddings.embed_query(chunk.page_content)

    records.append({
        "id": str(i),
        "content": chunk.page_content,
        "contentVector": vector,
        "source": chunk.metadata.get("source", ""),
        "page": chunk.metadata.get("page", 0),
        "department": chunk.metadata.get("department", "")
    })

search_client.upload_documents(records)
```

Production ingestion should normally batch uploads and use retries/error handling.

---

# 12. Hybrid Search

This is one of the most important parts.

Azure AI Search can execute:

```text
Keyword Search
       +
Vector Search
       ↓
RRF
       ↓
Unified Ranking
```

Azure AI Search uses **Reciprocal Rank Fusion (RRF)** to merge keyword and vector result rankings in hybrid queries. 

Example:

```python
from azure.search.documents.models import VectorizedQuery

query = "Can I carry forward unused vacation days?"

query_vector = embeddings.embed_query(query)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=20,
    fields="contentVector"
)

results = search_client.search(
    search_text=query,
    vector_queries=[vector_query],
    top=10,
    select=["id", "content", "source", "page"]
)

retrieved_docs = []

for result in results:
    retrieved_docs.append({
        "content": result["content"],
        "source": result["source"],
        "page": result["page"]
    })
```

This is a **hybrid search** because both:

```python
search_text=query
```

and:

```python
vector_queries=[vector_query]
```

are used.

---

# 13. Top-K

There are actually multiple K values you may use.

Example:

```text
Vector Search
     ↓
k = 20
     ↓
Hybrid ranking
     ↓
Semantic reranking
     ↓
top = 5
     ↓
LLM
```

You can think of it as:

```text
Retrieve broadly
       ↓
Rerank
       ↓
Send narrowly
```

Example:

```python
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=20,
    fields="contentVector"
)

results = search_client.search(
    search_text=query,
    vector_queries=[vector_query],
    top=5
)
```

Azure documents that hybrid search supports controlling the result set with `top` and vector `k`; semantic ranking can rerank up to 50 candidates. 

---

# 14. Semantic Reranking

Azure AI Search provides **semantic ranking** as a second-stage ranking mechanism.

Conceptually:

```text
Keyword Search ───┐
                  ├──→ RRF → Candidate Results
Vector Search ────┘
                         ↓
                  Semantic Ranker
                         ↓
                      Top 5
```

Microsoft describes semantic ranking as secondary ranking that promotes semantically relevant results from the initial result set. 

This is different from vector similarity.

### Vector Search

```text
Embedding similarity
```

### Semantic Ranker

```text
Language understanding
+
Query/document relevance
```

---

# 15. LangChain Retriever

You can expose Azure AI Search through LangChain.

Conceptually:

```python
from langchain_community.retrievers import AzureAISearchRetriever

retriever = AzureAISearchRetriever(
    content_key="content",
    top_k=5,
    index_name=index_name,
    service_name=os.environ["AZURE_SEARCH_SERVICE"],
    api_key=os.environ["AZURE_SEARCH_KEY"]
)
```

However, for **advanced Azure hybrid + semantic-ranking controls**, I prefer using the Azure Search SDK directly and wrapping it as a LangChain `BaseRetriever`/function. This gives you explicit control over:

- Vector `k`
- `top`
- Filters
- Hybrid search
- Semantic configuration
- Search scores
- Metadata

---

# 16. Context Builder

```python
def build_context(docs):

    return "\n\n".join(
        f"""
SOURCE: {doc['source']}
PAGE: {doc['page']}

{doc['content']}
"""
        for doc in docs
    )
```

Then:

```python
context = build_context(retrieved_docs)
```

---

# 17. RAG Prompt

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an enterprise HR assistant.

Answer the question ONLY using the provided context.

If the answer is not available in the context,
say: "I don't have enough information."

Always mention the source when possible.

Context:
{context}

Question:
{question}

Answer:
""")
```

---

# 18. Generate Answer

```python
chain = prompt | llm

response = chain.invoke({
    "context": context,
    "question": query
})

answer = response.content

print(answer)
```

Now your basic RAG is:

```text
Question
   ↓
Hybrid Search
   ↓
Top-K
   ↓
Context
   ↓
Prompt
   ↓
Azure OpenAI
   ↓
Answer
```

---

# 19. Add Azure Content Safety

Content Safety should be a **separate safety layer**, not your only security mechanism.

Azure AI Content Safety provides text/image APIs for detecting potentially harmful content. Its text API supports categories including Hate, SelfHarm, Sexual, and Violence. 

```python
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions
from azure.core.credentials import AzureKeyCredential

safety_client = ContentSafetyClient(
    endpoint=os.environ["CONTENT_SAFETY_ENDPOINT"],
    credential=AzureKeyCredential(
        os.environ["CONTENT_SAFETY_KEY"]
    )
)

def is_safe(text):

    result = safety_client.analyze_text(
        AnalyzeTextOptions(text=text)
    )

    for category in result.categories_analysis:

        if category.severity >= 4:
            return False

    return True
```

---

# 20. Input Safety

Before retrieval:

```python
if not is_safe(query):
    return "I can't assist with that request."
```

Architecture:

```text
User Question
      ↓
Content Safety
      ↓
Safe?
 ┌────┴────┐
No         Yes
│           │
Block       Search
```

---

# 21. Output Safety

After generation:

```python
if not is_safe(answer):
    answer = "I can't provide that response."
```

Full flow:

```text
User
 ↓
Input Safety
 ↓
Retrieval
 ↓
Azure OpenAI
 ↓
Output Safety
 ↓
Answer
```

Azure Content Safety is a content-risk control; it does **not** replace authentication, authorization, document-level access control, or tool permissions.

---

# 22. Complete Simple RAG Function

Now combine everything:

```python
def rag(query):

    # 1. Input safety
    if not is_safe(query):
        return "Request blocked by safety policy."

    # 2. Embed query
    query_vector = embeddings.embed_query(query)

    # 3. Hybrid search
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=20,
        fields="contentVector"
    )

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        top=5,
        select=["content", "source", "page"]
    )

    # 4. Build context
    docs = [
        {
            "content": r["content"],
            "source": r["source"],
            "page": r["page"]
        }
        for r in results
    ]

    context = build_context(docs)

    # 5. Generate
    response = chain.invoke({
        "context": context,
        "question": query
    })

    answer = response.content

    # 6. Output safety
    if not is_safe(answer):
        return "Generated response blocked by safety policy."

    return answer
```

---

# 23. One Important Improvement

The above is deliberately simple.

For production, the retrieval flow should be closer to:

```text
Query
 ↓
Input Safety
 ↓
Query Rewrite
 ↓
Hybrid Retrieval
 ↓
Top 20-50 Candidates
 ↓
Semantic Reranking
 ↓
Top 5
 ↓
Context Compression
 ↓
Prompt
 ↓
Azure OpenAI
 ↓
Groundedness / Safety
 ↓
Answer + Citations
```

Azure AI Search's semantic ranker can act as the second-stage reranker. 

---

# 24. Precision and Recall

Suppose the ground-truth relevant documents are:

```text
A, B, C
```

Retrieved:

```text
A, B, D, E, F
```

Relevant retrieved:

```text
A, B
```

### Precision@5

```text
Relevant Retrieved / Total Retrieved

= 2 / 5
= 0.40
```

### Recall@5

```text
Relevant Retrieved / Total Relevant

= 2 / 3
= 0.67
```

So:

| Metric | Measures |
|---|---|
| **Precision@K** | How much of retrieved content is relevant |
| **Recall@K** | How much of the relevant content was retrieved |

---

# 25. RAG Evaluation Dataset

Create a small golden dataset:

```python
evaluation_data = [
    {
        "question": "How many vacation days can be carried forward?",
        "ground_truth": "Employees can carry forward 5 vacation days."
    },
    {
        "question": "What is the maternity leave duration?",
        "ground_truth": "The maternity leave duration is 26 weeks."
    }
]
```

Then run your RAG:

```python
results = []

for item in evaluation_data:

    answer = rag(item["question"])

    results.append({
        "question": item["question"],
        "answer": answer,
        "ground_truth": item["ground_truth"]
    })
```

---

# 26. RAGAS

RAGAS provides RAG-specific evaluation metrics. Current RAGAS documentation includes metrics such as **Context Precision**, which evaluates whether relevant chunks are ranked higher than irrelevant chunks. 

Typical RAG evaluation:

```text
                 RAG Evaluation
                       │
          ┌────────────┴────────────┐
          ▼                         ▼
      Retrieval                  Generation
          │                         │
   Context Precision          Faithfulness
   Context Recall             Answer Relevance
```

Typical metrics:

| Metric | What it measures |
|---|---|
| Context Precision | Relevant chunks ranked higher |
| Context Recall | Relevant information retrieved |
| Faithfulness | Answer supported by context |
| Answer Relevance | Answer addresses question |

---

# 27. RAGAS Example

The exact RAGAS API varies by release, so conceptually:

```python
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy
)

dataset = {
    "user_input": [
        "How many vacation days can be carried forward?"
    ],
    "response": [
        "Employees can carry forward 5 vacation days."
    ],
    "retrieved_contexts": [
        [
            "Employees can carry forward 5 vacation days."
        ]
    ],
    "reference": [
        "Employees can carry forward 5 vacation days."
    ]
}
```

Then evaluate with the RAGAS API appropriate to the installed version.

The key interview point is **not the exact function signature**; it is understanding what each metric measures.

---

# 28. DeepEval

DeepEval is particularly useful for evaluating RAG pipelines and separating **retriever quality from generator quality**. Its current RAG metrics include contextual precision, contextual recall, contextual relevancy, answer relevancy, and faithfulness. 

Install:

```bash
pip install -U deepeval
```

Create a test case:

```python
from deepeval import evaluate
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="How many vacation days can be carried forward?",
    actual_output="Employees can carry forward 5 vacation days.",
    expected_output="Employees can carry forward 5 vacation days.",
    retrieval_context=[
        "Employees can carry forward 5 vacation days."
    ]
)
```

Metrics:

```python
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
    AnswerRelevancyMetric
)

metrics = [
    ContextualPrecisionMetric(),
    ContextualRecallMetric(),
    ContextualRelevancyMetric(),
    FaithfulnessMetric(),
    AnswerRelevancyMetric()
]

evaluate(
    test_cases=[test_case],
    metrics=metrics
)
```

DeepEval specifically recommends using the three contextual metrics for retrieval and **Faithfulness + Answer Relevancy** for the generator. 

---

# 29. What Each DeepEval Metric Tells You

```text
                  RAG
                   │
          ┌────────┴────────┐
          │                 │
      Retriever          Generator
          │                 │
          ▼                 ▼
Contextual              Faithfulness
Precision               Answer Relevancy
Contextual
Recall
Contextual
Relevancy
```

### Contextual Precision

> Are relevant chunks ranked higher than irrelevant chunks?

This is particularly useful for assessing your reranking quality. 

### Contextual Recall

> Did the retrieved context contain enough information to support the expected answer? 

### Contextual Relevancy

> Is the retrieved context actually relevant to the question? 

### Faithfulness

> Is the generated answer supported by the retrieved context?

### Answer Relevancy

> Does the answer actually address the user's question?

---

# 30. End-to-End Evaluation Architecture

```text
                   Test Dataset
                        │
                        ▼
                  RAG Pipeline
                        │
          ┌─────────────┴──────────────┐
          │                            │
          ▼                            ▼
      Retrieved                    Final Answer
       Chunks                          │
          │                            │
          ▼                            ▼
 Precision@K                      Faithfulness
 Recall@K                         Answer Relevancy
 Context Precision
 Context Recall
 Context Relevancy
```

---

# 31. Production-Grade Final Architecture

```text
                         USER
                           │
                           ▼
                  ┌─────────────────┐
                  │ Entra ID / Auth │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │ Content Safety  │
                  │ Input / Prompt  │
                  └────────┬────────┘
                           │
                           ▼
                    LangChain App
                           │
                           ▼
                    Query Rewrite
                           │
                           ▼
              ┌─────────────────────────┐
              │     Azure AI Search     │
              │                         │
              │ Keyword + Vector        │
              │        ↓                │
              │ RRF Hybrid Ranking      │
              │        ↓                │
              │ Semantic Reranker       │
              │        ↓                │
              │       Top-K             │
              └───────────┬─────────────┘
                          │
                          ▼
                   Context Builder
                          │
                          ▼
                   Azure OpenAI
                          │
                          ▼
                 Groundedness Check
                          │
                          ▼
                  Content Safety
                          │
                          ▼
                Answer + Citations
                          │
                          ▼
                         USER


        DOCUMENT INGESTION
                │
                ▼
        Blob / SharePoint
                │
                ▼
      Document Intelligence
                │
                ▼
        Structure + OCR
                │
                ▼
         Chunk + Metadata
                │
                ▼
           Embeddings
                │
                ▼
        Azure AI Search Index


        EVALUATION
                │
        ┌───────┴────────┐
        ▼                ▼
      RAGAS          DeepEval
        │                │
        ├─ Precision     ├─ Context Precision
        ├─ Recall        ├─ Context Recall
        ├─ Faithfulness  ├─ Context Relevancy
        └─ Relevance     ├─ Faithfulness
                         └─ Answer Relevancy
```

## Interview-ready explanation

> **"I would separate my Azure RAG system into ingestion, retrieval, generation, safety, and evaluation layers. During ingestion, I can use Blob Storage or SharePoint as the source, Document Intelligence for complex document extraction, chunk the content while preserving metadata, generate embeddings, and index it into Azure AI Search. At query time, I would perform hybrid keyword and vector retrieval, use semantic ranking as a second-stage reranker, retrieve an appropriate Top-K set, construct a grounded prompt, and call Azure OpenAI through LangChain. I would put Content Safety around the input and output paths, enforce identity and document-level authorization separately, and evaluate retrieval using Precision@K, Recall@K and contextual metrics, while evaluating generation using faithfulness and answer relevance through RAGAS or DeepEval."**

This architecture gives you a strong **end-to-end answer for the Altimetrik AI Consultant interview**, because it covers not just "how to build RAG," but also **retrieval quality, reranking, security, safety, observability/evaluation, and production concerns**.